# 4b · MIL training — instance-level, the instrument for Q2

> ⛔ **This stage is a stub and must stay one until the blank in §2 is filled.** The architecture is
> not built, and the criterion that decides whether it succeeded is deliberately written *before* any
> run exists. A structured-looking result read after the fact is not evidence
> ([TODO](../docs/TODO.md), *Agreed plan, Step 2*).

## What this notebook is for

**Q2 — does a model trained on single cells learn cellular heterogeneity implicitly?** It is the
clinically consequential question, because relapse is driven by rare surviving subpopulations rather
than by the average cell. It is **structurally unanswerable** under `4a_percell_training`'s per-cell model: every
cell of a line carries that line's label, so the objective penalises exactly the within-line variation
Q2 asks about ([Step 03](../docs/steps/03-model-and-training-design.md#every-cell-of-a-line-carries-the-identical-label)).

MIL is the smallest change that makes the question askable: a **bag of cells → one line label**
constrains only the aggregate, leaving the model free to differ between cells of the same line.

## Two decisions already taken (Selin, 12.08.2026)

**Instance-level, not attention pooling.** Every cell gets its own *predicted response*, and the line
prediction aggregates them — rather than every cell getting an attention *weight* over a pooled
embedding. The two are the standard MIL alternatives (Ilse, Tomczak & Welling, *Attention-based Deep
Multiple Instance Learning*, ICML 2018): embedding-level usually predicts better, instance-level is
readable at the level of the individual instance. Q2 is a question about readability, so the trade is
taken deliberately and the cost in predictive performance is expected.

⚠️ **One consequence, recorded so it is not rediscovered:** selecting "the top-k cells" *by their
predicted value* and scoring that subset against the line's true response is biased by construction —
the extremes are shifted away from the line mean because they were chosen for being extreme. The
subpopulation-predictivity test that would have used it is therefore **not** in the criterion below.
It becomes available only if an attention weight is added alongside the per-cell predictions.

**Same scorer as the per-cell model.** This notebook writes out-of-fold predictions in the shared
format — one row per cell line × drug × arm — so [`5_evaluation`](5_evaluation.ipynb) computes order,
top-of-order, values and spread for MIL and the per-cell model through identical code. The two are
comparable because they went through the same scorer, not because two notebooks agree by convention.

**Loss:** whatever `4a_percell_training` settles on, unchanged, so the architecture is the only thing that moves
([the governing rule](../docs/TODO.md)). Ranking losses (RankNet, LambdaRank) become well-posed *here*
and nowhere earlier — they need one score per cell line, which is what a bag produces — but they are a
second change and belong to a later run, not this one.

## 2 · What counts as a positive Q2 result — fixed before the run

**There is no ground truth for within-line heterogeneity of drug response.** Every label is one number
per (cell line, drug); no per-cell response was ever measured, and SCP542 carries no post-treatment
single-cell data, so the ideal test — do the model's resistant cells match the cells that actually
survive treatment — cannot be run here. This is a limitation of the data, not of the design, and it
belongs in the write-up rather than being left implicit.

What can be established is narrower and still worth having: **that the model's per-cell predictions
vary within a line, that the variation is reproducible rather than noise, and that it is not a
sequencing artifact.**

Four stages, in order. Each has a distinct role — two are conditions, one is the test, one is a veto.

| # | Stage | Role | Passes when |
|---|---|---|---|
| **7** | **Synthetic positive control** — bags mixed from two cell lines of known, different response, labelled with the mixture-weighted value | **precondition** | the model recovers the known mixture to within the threshold below |
| **1** | **Spread** — within-line standard deviation of per-cell predicted responses | necessary condition | spread is a stated fraction of what stage 7 produced |
| **2** | **Reproducibility** — do independent seeds assign high and low predictions to the *same* cells? | **the test** | per-cell agreement across seeds exceeds the shuffled-cell control, by a stated fraction of stage 7's agreement |
| **6** | **Confound regression** — per-cell predictions against total counts, genes detected, mitochondrial fraction and cell-cycle score | **veto** | the confounds do *not* explain the variation |

**Why stage 7 comes first.** Without it a negative result is uninterpretable — "no heterogeneity found"
cannot be distinguished from "this method cannot find heterogeneity". With it, a negative becomes a
result: no detectable heterogeneity, by an instrument demonstrated to detect it when present.

**Why stage 6 is a veto and not an analysis.** It looks descriptive, but it can turn a pass into a
fail: predictions that replicate across seeds *and* are explained by library size are a sequencing
artifact, not biology. Pre-registered here so it cannot become something run only when the answer is
unwelcome.

**Why only stage 7 needs a number chosen by judgement.** Stage 7 has ground truth, so it shows what
spread and what cross-seed agreement look like when heterogeneity is *definitely* present. Stages 1 and
2 are then expressed as fractions of that, instead of thresholds invented in advance.

> ⬜ **BLANK — Selin's, and the last thing needed before this notebook may be written.**
>
> How well must the model recover a known mixture in stage 7 before the instrument counts as working?
>
> `Q2_CONTROL_THRESHOLD = ...`
>
> Whatever is chosen goes here with its reasoning, and the stage-1 and stage-2 fractions with it.

## 3 · Closing analysis — what kind of cells were they?

Short and descriptive, and **it gates nothing**. By the time it runs, §2 has already decided whether Q2
is positive. This section says *what was found*, not *whether* something was found — the distinction
matters, because an enrichment discovered here cannot be promoted into evidence afterwards.

Two figures and a table:

**a · What the predictions track (stage 6, reported rather than vetoing).** The same regression the veto
uses, shown rather than thresholded: how much of the within-line variation in per-cell predictions is
explained by each of total counts, genes detected, mitochondrial fraction and cell-cycle score. If the
veto passed, these are all small, and showing them is what makes that credible.

**b · Which annotated programs the predictions track (stage 3).** Kinker et al. 2020 annotated recurrent
heterogeneity programs for this exact dataset, independently of any drug-response label. Both sides are
continuous — each cell has a program score, and instance-level MIL gives each cell a predicted response —
so **correlate the two across a line's cells**, per program, against a within-line permutation null.

*No top-k, decided 12.08.2026 (Selin).* An earlier draft took the most- and least-resistant predicted
cells and tested them for enrichment. Correlating the full continuous signal is strictly better here: it
needs no `k` to justify, uses every cell instead of a slice, and is the same method as (a) above — so the
confound check and the biology check are read on one scale rather than two.

⚠️ **Read the enrichment carefully.** Cell-cycle enrichment is close to guaranteed and would be weak
evidence of anything: the project already refuted *the cell-line effect is largely proliferation*
([Corrections](../docs/steps/corrections-and-dead-ends.md#the-cell-line-effect-is-largely-proliferation)),
and Kinker's two named associations are recorded as **not transferring to this task**
([Dead ends](../docs/steps/corrections-and-dead-ends.md#kinkers-two-named-associations-do-not-transfer-to-this-task)).
A program *other* than cell cycle would be the interesting outcome.

**c · One table** — per drug: does the cell ordering repeat across drugs, or is it drug-specific? A
general axis and a drug-specific subpopulation are different findings, and the table is the cheapest
way to tell them apart.

---

### What must not happen before the blank in §2 is filled

No cells are added below this one. Building the model first and choosing the criterion afterwards is
the failure mode this stub exists to prevent, and it is the reason the criterion is written here in
prose rather than left to the run that will be scored by it.